# 0. Libraries 

In [ ]:
# ==============================
# Core libraries
# ==============================
import numpy as np
import pandas as pd
import time
import warnings
import multiprocessing
import re
from pathlib import Path
from datetime import date, datetime, timedelta

import requests
import certifi

# Use all but one CPU core for parallel processing
num_cores = max(multiprocessing.cpu_count() - 1, 1)
print("Using", num_cores, "cores for parallel processing.")

warnings.filterwarnings("ignore")


# ==============================
# Visualisation
# ==============================
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")  # optional, just to make plots look nicer


# ==============================
# Statistical utilities (EDA, tests)
# ==============================
import scipy.stats as stats
from scipy.stats import chi2, chi2_contingency, f_oneway

# Statsmodels (OLS, ANOVA, etc.)
import statsmodels.api as sm
from statsmodels.formula.api import ols


# ==============================
# Preprocessing & feature engineering
# ==============================
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    RobustScaler,
    OneHotEncoder
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Optional: dimensionality reduction
from sklearn.decomposition import PCA


# ==============================
# Modelling algorithms (base models)
# ==============================

# Linear models / GLM-style
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

# Tree-based and ensemble models
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    BaggingRegressor,
    StackingRegressor,
    VotingRegressor
)

from sklearn.tree import DecisionTreeRegressor

# Distance-based & kernel-based models
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# Neural network regressor
from sklearn.neural_network import MLPRegressor


# ==============================
# Gradient boosting libraries (external)
# ==============================

# LightGBM
try:
    import lightgbm as lgb
    lgb_available = True
    print("LightGBM available.")
except ImportError:
    lgb_available = False
    print("LightGBM NOT available (install lightgbm if you want to use it).")

# CatBoost (optional; often slower but nice to try)
try:
    import catboost as cb
    cb_available = True
    print("CatBoost available.")
except ImportError:
    cb_available = False
    print("CatBoost NOT available (install catboost if you want to use it).")


# ==============================
# Model evaluation & selection
# ==============================
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ==============================
# Model interpretation tools
# ==============================
from sklearn.inspection import (
    permutation_importance,
    PartialDependenceDisplay
)

# If you later want SHAP, you can add:
# import shap


# 1. Loading the Data & Basic Inspection

In [ ]:
# Importing the training dataset
train = pd.read_csv("ML_WP_data/train.csv")

# Basic structural info (includes shape, dtypes, non-null counts)
train.info()

# Quick look at the first rows
display(train.head())

# Calculate total number of NaN values in the DataFrame
total_train_nans = train.isna().sum().sum()
print("Total NaN values in the training DataFrame:", total_train_nans)

# ------------------------------
# Helper: missingness summary (train only)
# ------------------------------

def missing_summary(df, sort_by="Percent_Missing", ascending=False):
    """
    Build a table with:
    - Data type
    - Number of missing values
    - Percentage of missing values
    - Basic descriptive stats (mean, std, min, 25%, 50%, 75%, max) for numeric cols
    """

    n_rows = len(df)

    # Core missing-value info
    miss = df.isna().sum()
    miss = miss[miss > 0]  # keep only variables with at least one missing

    summary = pd.DataFrame({
        "Data_Type": df[miss.index].dtypes.astype(str),
        "Missing_Values": miss,
        "Percent_Missing": (miss / n_rows * 100).round(2)
    })

    # Descriptive stats for numeric columns
    numeric_cols = df[miss.index].select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        desc = df[numeric_cols].describe().T[
            ["mean", "std", "min", "25%", "50%", "75%", "max"]
        ]
        summary = summary.join(desc, how="left")

    # Sort and print total
    summary = summary.sort_values(sort_by, ascending=ascending)
    print(f"Total variables with missing values: {summary.shape[0]}")

    return summary

# Compute missingness summary for train
missing_train = missing_summary(train)


# 2. Target columns definition

In [ ]:
target_cols = [
    "target_tre200h0_plus12h",
    "target_tre200h0_plus24h",
    "target_tre200h0_plus48h"
]
print("Target columns:", target_cols)
print("Missing values per target:")
print(train[target_cols].isna().sum(), "\n")


# 3. Exploratory Data Analysis (EDA)

## 3.1 Missingness & descriptive statistics

In [ ]:
# Summary for variables with missing values
missing_summary = (
    pd.DataFrame({
        "Data_Type": train.dtypes,
        "Missing_Values": train.isnull().sum(),
        "Percent_Missing": (train.isnull().sum() / len(train) * 100).round(2)
    })
    .query("Missing_Values > 0")
    .sort_values(by="Missing_Values", ascending=False)
)

# Descriptive stats
summary_stats = train.describe(include="all").transpose()

# Merge missingness with basic stats
merged_summary = missing_summary.merge(
    summary_stats[["mean", "std", "min", "25%", "50%", "75%", "max"]],
    left_index=True,
    right_index=True,
    how="left"
)

print(merged_summary.to_string())
print(f"\nTotal variables with missing values: {len(missing_summary)}")


## 3.2 Outlier overview (numeric variables, excluding targets)

In [ ]:
# Numeric columns (excluding targets)
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in target_cols]

def detect_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    return outliers

outlier_summary = {}
for col in numeric_cols:
    n_outliers = len(detect_outliers(train, col))
    outlier_summary[col] = n_outliers

outlier_df = pd.DataFrame(list(outlier_summary.items()), columns=["Variable", "Outlier_Count"])
outlier_df["Outlier_%"] = (outlier_df["Outlier_Count"] / len(train)) * 100
outlier_df.sort_values(by="Outlier_%", ascending=False, inplace=True)

display(outlier_df.head(20))


## 3.3 Target distributions (histogram, boxplot, Q-Q plot)

In [ ]:
fig, axes = plt.subplots(len(target_cols), 3, figsize=(15, 12))

for i, target_col in enumerate(target_cols):
    # Histogram
    axes[i, 0].hist(train[target_col].dropna(), bins=50, edgecolor="black", alpha=0.7)
    axes[i, 0].set_title(f"{target_col} – Distribution")
    axes[i, 0].set_xlabel("Temperature (°C)")
    axes[i, 0].set_ylabel("Frequency")

    # Boxplot
    axes[i, 1].boxplot(train[target_col].dropna(), vert=True)
    axes[i, 1].set_title(f"{target_col} – Boxplot")
    axes[i, 1].set_ylabel("Temperature (°C)")

    # Q–Q plot vs normal
    stats.probplot(train[target_col].dropna(), dist="norm", plot=axes[i, 2])
    axes[i, 2].set_title(f"{target_col} – Q–Q Plot vs Normal")

plt.tight_layout()
plt.show()


## 3.4 Skewness and kurtosis of targets

In [ ]:
for t in target_cols:
    clean_series = train[t].dropna()
    clean_series = clean_series[np.isfinite(clean_series)]

    skew = stats.skew(clean_series)
    kurt = stats.kurtosis(clean_series)

    print(f"{t}: Skewness = {skew:.3f}, Kurtosis = {kurt:.3f}")


## 3.5 Missingness by hour (pattern + chi-square test)

In [ ]:
# Row-wise missing count
train["missing_count"] = train.isnull().sum(axis=1)

# Total missing values per hour
missing_by_hour = (
    train.groupby("hour")["missing_count"]
    .sum()
    .sort_values(ascending=False)
)

print("Top hours with most missing values:\n")
print(missing_by_hour.head(10))

# Plot
plt.figure(figsize=(10, 4))
missing_by_hour.sort_index().plot(kind="bar", edgecolor="black")
plt.title("Total Missing Values by Hour of the Day")
plt.xlabel("Hour (0–23)")
plt.ylabel("Number of Missing Values")
plt.tight_layout()
plt.show()

# Chi-square goodness-of-fit: are missing values equally distributed by hour?
missing_by_hour = train.groupby("hour")["missing_count"].sum()
expected = [missing_by_hour.sum() / len(missing_by_hour)] * len(missing_by_hour)

chi2_stat = ((missing_by_hour - expected) ** 2 / expected).sum()
p_value = 1 - chi2.cdf(chi2_stat, df=len(missing_by_hour) - 1)

print(f"Chi-square statistic: {chi2_stat:.2f}")
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
    print("→ Missingness differs significantly by hour (reject H₀).")
else:
    print("→ No significant hourly difference (fail to reject H₀).")


## 3.6 Missingness by season (if available) + chi-square + ANOVA

In [ ]:
# Assuming `train` is your DataFrame
if "season" in train.columns:
    # Use lower-case if that's how season is coded in the data
    season_order = ["winter", "spring", "summer", "autumn"]

    # Group by season and compute the total missing count
    missing_by_season = (
        train.groupby("season")["missing_count"]
        .sum()
        .reindex(season_order)
    )

    print("\nMissing values by season:")
    print(missing_by_season)

    # Plotting the missing values by season
    plt.figure(figsize=(6, 4))
    missing_by_season.plot(kind="bar", color="darkorange", edgecolor="black")
    plt.title("Missing Values by Season")
    plt.xlabel("Season")
    plt.ylabel("Total Missing Values")
    plt.xticks(rotation=45)  # Rotate x labels for better visibility
    plt.tight_layout()
    plt.show()

    # Chi-square test
    missing_by_season_noorder = train.groupby("season")["missing_count"].sum()
    total_missing = missing_by_season_noorder.sum()
    n_seasons = len(missing_by_season_noorder)
    expected = [total_missing / n_seasons] * n_seasons

    chi2_stat = ((missing_by_season_noorder - expected) ** 2 / expected).sum()
    df_chi = n_seasons - 1
    p_value = 1 - chi2.cdf(chi2_stat, df=df_chi)

    print("=== Chi-square Test for Seasonal Missingness ===")
    print(f"Chi-square statistic: {chi2_stat:.2f}")
    print(f"Degrees of freedom: {df_chi}")
    print(f"p-value: {p_value:.4f}")

    if p_value < 0.05:
        print("→ Missingness differs significantly by season (reject H₀).")
    else:
        print("→ No significant seasonal difference in missingness (fail to reject H₀).")

    # ANOVA: does mean missing_count differ by season?
    model = ols("missing_count ~ C(season)", data=train).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)

    print("\n=== ANOVA Test for Seasonal Missingness ===")
    print(anova_table)

else:
    print("Column 'season' not found in 'train': skipping seasonal missingness analysis.")

## 3.7 Distributions of variables with most outliers

In [ ]:
# Take the top k variables with highest outlier percentage
k = 13  # adjust as you like
top_outlier_vars = outlier_df.head(k)["Variable"].tolist()

print("Variables with highest outlier %:")
print(top_outlier_vars)

for col in top_outlier_vars:
    series = train[col].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"Outlier inspection – {col}", fontsize=12)

    # Histogram
    axes[0].hist(series, bins=50, edgecolor="black", alpha=0.7)
    axes[0].set_title("Histogram")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Frequency")

    # Boxplot
    axes[1].boxplot(series, vert=True)
    axes[1].set_title("Boxplot")
    axes[1].set_ylabel(col)

    plt.tight_layout()
    plt.show()


### log10 histograms for non-negative, highly skewed variables among top outliers

In [ ]:
log_candidates = []
for col in top_outlier_vars:
    s = train[col].dropna()
    if (s >= 0).all() and s.max() > 0:
        log_candidates.append(col)

print("Log-scale candidates (non-negative among top outliers):")
print(log_candidates)

for col in log_candidates:
    s = train[col].dropna()
    s_pos = s[s > 0]  # avoid log(0)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"Raw vs log10 – {col}", fontsize=12)

    axes[0].hist(s, bins=50, edgecolor="black", alpha=0.7)
    axes[0].set_title("Raw scale")
    axes[0].set_xlabel(col)

    axes[1].hist(np.log10(s_pos), bins=50, edgecolor="black", alpha=0.7)
    axes[1].set_title("log10 scale (values > 0)")
    axes[1].set_xlabel(f"log10({col})")

    plt.tight_layout()
    plt.show()


# 1. Quickly see suspicious min/max per variable group

In [ ]:
def aggregate_min_max_by_variable(train):
    summary = []

    for col in train.select_dtypes(include="number").columns:
        # Split column name
        parts = col.split("_")

        # If this is a lag variable, keep full name
        if "lag" in col:
            base_var = col
        else:
            base_var = parts[0]  # e.g., tre200h0, ure200h0, etc.

        summary.append({
            "variable": base_var,
            "min": train[col].min(),
            "max": train[col].max()
        })

    summary_df = pd.DataFrame(summary)

    # Aggregate across stations
    aggregated = (
        summary_df
        .groupby("variable", as_index=False)
        .agg(min=("min", "min"), max=("max", "max"))
        .sort_values("variable")
    )

    return aggregated

var_min_max = aggregate_min_max_by_variable(train)
var_min_max


In [ ]:
def clean_weather_outliers(train):
    train = train.copy()

    # Just in case – very loose physical bounds
    temp_cols = [c for c in train.columns if c.startswith("tre200h0")]
    for c in temp_cols:
        train.loc[(train[c] < -40) | (train[c] > 50), c] = np.nan

    hum_cols = [c for c in train.columns if c.startswith("ure200h0")]
    for c in hum_cols:
        train.loc[(train[c] < 0) | (train[c] > 100), c] = np.nan

    precip_cols = [c for c in train.columns if c.startswith("rre150h0")]
    for c in precip_cols:
        train.loc[train[c] < 0, c] = np.nan

    rad_cols = [c for c in train.columns if c.startswith("gre000h0")]
    for c in rad_cols:
        train.loc[train[c] < 0, c] = np.nan

    wind_cols = [c for c in train.columns if c.startswith("fkl010h0") or c.startswith("fkl010h3")]
    for c in wind_cols:
        train.loc[train[c] < 0, c] = np.nan

    return train


# Missing Values Analysis


In [ ]:
missing_summary_raw = (
    train
    .isna()
    .sum()
    .loc[lambda x: x > 0]
    .sort_values(ascending=False)
)

missing_summary_raw


In [ ]:
variable_groups = {
    "temperature": [c for c in train.columns if c.startswith("tre200h0")],
    "humidity":    [c for c in train.columns if c.startswith("ure200h0")],
    "pressure":    [c for c in train.columns if c.startswith("prestah0")],
    "wind":        [c for c in train.columns if c.startswith("fkl010h0") or c.startswith("fkl010h3")],
    "precipitation": [c for c in train.columns if c.startswith("rre150h0")],
    "radiation":   [c for c in train.columns if c.startswith("gre000h0")],
}

missing_by_variable = {}

for var_name, cols in variable_groups.items():
    if len(cols) > 0:
        # Calculate the fraction of missing values
        missing_pct = train[cols].isna().mean().mean() * 100  # Use .mean() to get a single value
        missing_by_variable[var_name] = missing_pct

missing_pct_aggregated = (
    pd.Series(missing_by_variable)
      .sort_values(ascending=False)
)

print(missing_pct_aggregated)


In [ ]:
# Define column groups based on their prefixes using the 'train' DataFrame
temp_cols   = [c for c in train.columns if c.startswith("tre200h0")]
hum_cols    = [c for c in train.columns if c.startswith("ure200h0")]
pres_cols   = [c for c in train.columns if c.startswith("prestah0")]
wind_cols   = [c for c in train.columns if c.startswith("fkl010h0") or c.startswith("fkl010h3")]
precip_cols = [c for c in train.columns if c.startswith("rre150h0")]
rad_cols    = [c for c in train.columns if c.startswith("gre000h0")]

# plotting the distributions

In [ ]:
def plot_pooled_distribution(df, cols, title, bins=50):
    values = pd.concat([df[c] for c in cols], axis=0).dropna()
    plt.figure()
    plt.hist(values, bins=bins)
    plt.title(title)
    plt.xlabel("Value")
    plt.ylabel("Frequency")
    plt.show()

# Define your column groups using the 'train' DataFrame
temp_cols   = [c for c in train.columns if c.startswith("tre200h0")]
hum_cols    = [c for c in train.columns if c.startswith("ure200h0")]
pres_cols   = [c for c in train.columns if c.startswith("prestah0")]
wind_cols   = [c for c in train.columns if c.startswith("fkl010h0") or c.startswith("fkl010h3")]
precip_cols = [c for c in train.columns if c.startswith("rre150h0")]
rad_cols    = [c for c in train.columns if c.startswith("gre000h0")]
sun_cols    = [c for c in train.columns if c.startswith("sre000h0")]

# Plot distributions for each variable group
plot_pooled_distribution(train, temp_cols,   "Temperature (all stations)")
plot_pooled_distribution(train, hum_cols,    "Humidity (all stations)")
plot_pooled_distribution(train, pres_cols,   "Pressure (all stations)")
plot_pooled_distribution(train, wind_cols,   "Wind speed (all stations)")
plot_pooled_distribution(train, precip_cols, "Precipitation (all stations)")
plot_pooled_distribution(train, rad_cols,    "Radiation (all stations)")
plot_pooled_distribution(train, sun_cols,    "Sunshine duration (all stations)")


| Variable family / prefix | Distribution shape | Key evidence from data | Correct imputation |
|--------------------------|--------------------|------------------------|--------------------|
| `tre200h0_*`, `tre200h0_lag24h` (temperature) | Approximately symmetric | Mean ≈ median, no mass at zero, moderate tails | Mean |
| `ure200h0_*` (humidity) | Bounded, mildly right-skewed | Median > mean, values in [0,100] | Median |
| `prestah0_*` (pressure) | Symmetric, very low variance | Mean ≈ median, tight IQR | Mean |
| `fkl010h0_*`, `fkl010h3_*` (wind) | Right-skewed | Long upper tail, median ≪ mean | Median |
| `rre150h0_*` (precipitation) | Zero-inflated, heavy right tail | Median = 0, many zeros, rare spikes | Zero |
| `gre000h0_*` (radiation) | Zero-inflated, very heavy right tail | Median ≈ 0, large max values | Zero |
| `sre000h0_*` (sunshine duration) | Zero-inflated | Median = 0, bounded [0,60] | Zero |


## Imputing missing values

In [ ]:
def impute_raw_weather(train):
    train = train.copy()

    # Temperature → mean
    temp_cols = [c for c in train.columns if c.startswith("tre200h0")]
    for c in temp_cols:
        train[c] = train[c].fillna(train[c].mean())

    # Humidity → median
    hum_cols = [c for c in train.columns if c.startswith("ure200h0")]
    for c in hum_cols:
        train[c] = train[c].fillna(train[c].median())

    # Pressure → mean
    pres_cols = [c for c in train.columns if c.startswith("prestah0")]
    for c in pres_cols:
        train[c] = train[c].fillna(train[c].mean())

    # Pressure tendency → median
    ppt_cols = [c for c in train.columns if c.startswith("pp0qffh0")]
    for c in ppt_cols:
        train[c] = train[c].fillna(train[c].median())

    # Wind → median
    wind_cols = [c for c in train.columns if c.startswith("fkl010h0") or c.startswith("fkl010h3")]
    for c in wind_cols:
        train[c] = train[c].fillna(train[c].median())

    # Precipitation → zero
    precip_cols = [c for c in train.columns if c.startswith("rre150h0")]
    for c in precip_cols:
        train[c] = train[c].fillna(0.0)

    # Radiation → zero
    rad_cols = [c for c in train.columns if c.startswith("gre000h0")]
    for c in rad_cols:
        train[c] = train[c].fillna(0.0)

    # Sunshine duration → zero
    sun_cols = [c for c in train.columns if c.startswith("sre000h0")]
    for c in sun_cols:
        train[c] = train[c].fillna(0.0)

    # Hour → mode
    if "hour" in train.columns:
        train["hour"] = train["hour"].fillna(train["hour"].mode()[0])

    # Season → mode
    if "season" in train.columns:
        train["season"] = train["season"].fillna(train["season"].mode()[0])

    return train


In [ ]:
# 1. Define target columns ONCE
target_cols = [c for c in train.columns if c.startswith("target_")]

# 2. Impute raw weather data
df_imputed = impute_raw_weather(train)

# 3. Drop columns that are entirely missing (defensive step)
df_imputed = df_imputed.dropna(axis=1, how="all")

# 4. Sanity check: no missing values in INPUT FEATURES
assert (
    df_imputed.drop(columns=target_cols).isna().sum().sum() == 0
), "There are still missing values in input features"

print(
    f"✓ Imputation successful: dataset has {df_imputed.shape[0]} rows "
    f"and {df_imputed.shape[1]} columns; input features contain no missing values."
)

# Creating Target-Specific Training datasets 
Targets are NOT imputed; rows with missing targets are dropped

In [ ]:
# 12-hour forecast dataset
df_train_12h = train.dropna(
    subset=["target_tre200h0_plus12h"]
).copy()

# 24-hour forecast dataset
df_train_24h = train.dropna(
    subset=["target_tre200h0_plus24h"]
).copy()

# 48-hour forecast dataset
df_train_48h = train.dropna(
    subset=["target_tre200h0_plus48h"]
).copy()


# Preprocessing

## Missingness indicators (add before imputing)

In [ ]:
FAMILY_PREFIXES = {
    "miss_tre200h0":   ("tre200h0",),
    "miss_ure200h0":   ("ure200h0",),
    "miss_prestah0":   ("prestah0",),
    "miss_pp0qffh0":   ("pp0qffh0",),
    "miss_wind":       ("fkl010h0", "fkl010h3"),
    "miss_rre150h0":   ("rre150h0",),
    "miss_gre000h0":   ("gre000h0",),
    "miss_sre000h0":   ("sre000h0",),
}

def add_missingness_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Family-level missingness flags
    for flag_name, prefixes in FAMILY_PREFIXES.items():
        cols = [c for c in df.columns if any(c.startswith(p) for p in prefixes)]
        if cols:
            df[flag_name] = df[cols].isna().any(axis=1).astype(int)
        else:
            df[flag_name] = 0

    # Hour/season flags (if present)
    if "hour" in df.columns:
        df["miss_hour"] = df["hour"].isna().astype(int)
    if "season" in df.columns:
        df["miss_season"] = df["season"].isna().astype(int)

    return df


## Cyclical hour + one-hot season + hour–season interactions

In [ ]:
def add_time_features_and_interactions(df: pd.DataFrame, drop_hour: bool = True) -> pd.DataFrame:
    df = df.copy()

    # --- Hour -> sin/cos ---
    if "hour" in df.columns:
        h = df["hour"].round().astype("Int64")  # keeps NaN if present
        h = h.clip(lower=0, upper=23)
        # temporary float for trig (NaN-safe)
        hf = h.astype(float)
        df["hour_sin"] = np.sin(2 * np.pi * hf / 24.0)
        df["hour_cos"] = np.cos(2 * np.pi * hf / 24.0)
        if drop_hour:
            df.drop(columns=["hour"], inplace=True)

    # --- Season -> one-hot ---
    if "season" in df.columns:
        df["season"] = df["season"].astype(str).str.strip().str.lower()
        df = pd.get_dummies(df, columns=["season"], prefix="season", drop_first=True)

    # --- Hour × season interactions (for linear models) ---
    season_dummy_cols = [c for c in df.columns if c.startswith("season_")]
    if "hour_sin" in df.columns and "hour_cos" in df.columns and season_dummy_cols:
        for s in season_dummy_cols:
            df[f"{s}_x_hour_sin"] = df[s] * df["hour_sin"]
            df[f"{s}_x_hour_cos"] = df[s] * df["hour_cos"]

    return df


## Imputation function 

In [ ]:
def impute_raw_weather(train: pd.DataFrame) -> pd.DataFrame:
    train = train.copy()

    # Temperature → mean
    temp_cols = [c for c in train.columns if c.startswith("tre200h0")]
    for c in temp_cols:
        train[c] = train[c].fillna(train[c].mean())

    # Humidity → median
    hum_cols = [c for c in train.columns if c.startswith("ure200h0")]
    for c in hum_cols:
        train[c] = train[c].fillna(train[c].median())

    # Pressure (station) → mean
    pres_cols = [c for c in train.columns if c.startswith("prestah0")]
    for c in pres_cols:
        train[c] = train[c].fillna(train[c].mean())

    # Sea-level pressure → median (as you did)
    ppt_cols = [c for c in train.columns if c.startswith("pp0qffh0")]
    for c in ppt_cols:
        train[c] = train[c].fillna(train[c].median())

    # Wind → median
    wind_cols = [c for c in train.columns if c.startswith("fkl010h0") or c.startswith("fkl010h3")]
    for c in wind_cols:
        train[c] = train[c].fillna(train[c].median())

    # Precipitation → zero
    precip_cols = [c for c in train.columns if c.startswith("rre150h0")]
    for c in precip_cols:
        train[c] = train[c].fillna(0.0)

    # Radiation → zero
    rad_cols = [c for c in train.columns if c.startswith("gre000h0")]
    for c in rad_cols:
        train[c] = train[c].fillna(0.0)

    # Sunshine duration → zero
    sun_cols = [c for c in train.columns if c.startswith("sre000h0")]
    for c in sun_cols:
        train[c] = train[c].fillna(0.0)

    # Hour → mode
    if "hour" in train.columns:
        train["hour"] = train["hour"].fillna(train["hour"].mode()[0])

    # Season → mode
    if "season" in train.columns:
        train["season"] = train["season"].fillna(train["season"].mode()[0])

    return train


## Building two datasets per horizon: (A) complete-case dropna, (B) imputed

In [ ]:
def make_horizon_datasets(train: pd.DataFrame, target_col: str):
    base = train.dropna(subset=[target_col]).copy()

    # A) Complete-case (drop any remaining NaNs in predictors)
    df_dropna = base.dropna().copy()

    # B) Imputed (your rules) + missingness indicators + time encodings + interactions
    df_imputed = add_missingness_indicators(base)
    df_imputed = impute_raw_weather(df_imputed)
    df_imputed = add_time_features_and_interactions(df_imputed, drop_hour=True)

    return df_dropna, df_imputed

df24_dropna, df24_imputed = make_horizon_datasets(train, "target_tre200h0_plus24h")
df12_dropna, df12_imputed = make_horizon_datasets(train, "target_tre200h0_plus12h")
df48_dropna, df48_imputed = make_horizon_datasets(train, "target_tre200h0_plus48h")

# Reusable evaluation utilities (config, models, evaluation, MAE comparison)

In [ ]:
# ============================================================
# Run ALL models on BOTH datasets (drop_na vs imputed)
# Includes: Ridge, Lasso, ElasticNet, RF, ExtraTrees, GB
# Optional: XGBoost, LightGBM (if available flags are True)
# ============================================================


# ----------------------------
# 1) Horizon mapping
# ----------------------------
HORIZONS = {
    "12h": "target_tre200h0_plus12h",
    "24h": "target_tre200h0_plus24h",
    "48h": "target_tre200h0_plus48h",
}

# ----------------------------
# 2) Models (exactly as you requested)
# ----------------------------
def build_models(random_state: int = 42, num_cores: int = -1) -> dict:
    models = {
        "Ridge": Ridge(),
        "Lasso": Lasso(max_iter=5000),
        "ElasticNet": ElasticNet(max_iter=5000),

        "RandomForest": RandomForestRegressor(
            n_estimators=200,
            random_state=random_state,
            n_jobs=num_cores,
            criterion="absolute_error",
        ),
        "ExtraTrees": ExtraTreesRegressor(
            n_estimators=200,
            random_state=random_state,
            n_jobs=num_cores,
        ),
        "GradientBoosting": GradientBoostingRegressor(random_state=random_state),
    }

    # Optional: XGBoost
    if globals().get("XGBOOST_AVAILABLE", False):
        models["XGBRegressor"] = xgb.XGBRegressor(
            objective="reg:squarederror",
            n_estimators=300,
            random_state=random_state,
            n_jobs=num_cores if num_cores != -1 else None,
        )

    # Optional: LightGBM
    if globals().get("LIGHTGBM_AVAILABLE", False):
        models["LGBMRegressor"] = lgb.LGBMRegressor(
            n_estimators=500,
            objective="regression",
            random_state=random_state,
            n_jobs=num_cores,
        )

    return models

# ----------------------------
# 3) Evaluate all models on ONE dataset
# ----------------------------
def evaluate_models_on_dataset(
    df_model: pd.DataFrame,
    target_col: str,
    horizons: dict,
    models: dict,
    dataset_name: str,
    test_size: float = 0.2,
    random_state: int = 42
) -> pd.DataFrame:

    df_model = df_model.copy()
    y = df_model[target_col]

    # drop ALL target columns to avoid leakage
    target_cols_present = [c for c in horizons.values() if c in df_model.columns]
    X = df_model.drop(columns=target_cols_present, errors="ignore")

    # numeric features only
    numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", RobustScaler()),
            ]), numeric_features)
        ],
        remainder="drop"
    )

    # random split (data not time-ordered)
    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    rows = []
    for name, reg in models.items():
        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("regressor", reg)
        ])

        t0 = time.time()
        pipe.fit(X_train, y_train)
        train_time = time.time() - t0

        pred = pipe.predict(X_valid)
        mae = mean_absolute_error(y_valid, pred)
        rmse = mean_squared_error(y_valid, pred) ** 0.5
        r2 = r2_score(y_valid, pred)

        rows.append({
            "dataset": dataset_name,
            "model": name,
            "mae": mae,
            "rmse": rmse,
            "r2": r2,
            "train_time_sec": train_time
        })

    return pd.DataFrame(rows)

# ----------------------------
# 4) Evaluate all models on BOTH datasets + MAE pivot
# ----------------------------
def evaluate_on_both_datasets(
    datasets: dict,          # {"drop_na": df24_dropna, "imputed": df24_imputed}
    target_col: str,
    horizons: dict,
    models: dict,
    test_size: float = 0.2,
    random_state: int = 42
) -> pd.DataFrame:
    out = []
    for ds_name, df_ds in datasets.items():
        out.append(
            evaluate_models_on_dataset(
                df_model=df_ds,
                target_col=target_col,
                horizons=horizons,
                models=models,
                dataset_name=ds_name,
                test_size=test_size,
                random_state=random_state
            )
        )
    return pd.concat(out, ignore_index=True)

def mae_comparison_table(results_df: pd.DataFrame) -> pd.DataFrame:
    pivot = results_df.pivot_table(index="model", columns="dataset", values="mae", aggfunc="mean")
    if {"imputed", "drop_na"}.issubset(pivot.columns):
        pivot["imputed_minus_dropna"] = pivot["imputed"] - pivot["drop_na"]
        pivot = pivot.sort_values("imputed_minus_dropna")
    return pivot

# ----------------------------
# 5) RUN (24h example)
# ----------------------------
datasets_24h = {
    "drop_na": df24_dropna.copy(),
    "imputed": df24_imputed.copy(),
}

target_col = HORIZONS["24h"]

models = build_models(
    random_state=42,
    num_cores=globals().get("num_cores", -1)
)

results_df = evaluate_on_both_datasets(
    datasets=datasets_24h,
    target_col=target_col,
    horizons=HORIZONS,
    models=models,
    test_size=0.2,
    random_state=42
)

display(results_df.sort_values(["dataset", "mae"]).round(4))
display(mae_comparison_table(results_df).round(4))


# Run evaluation for a chosen horizon and compare drop_na vs imputed

In [ ]:
# Pick horizon + target column
current_horizon = "24h"
target_col = HORIZONS[current_horizon]

# Build datasets dict
datasets = {
    "drop_na": df24_dropna.copy(),
    "imputed": df24_imputed.copy(),
}

# Build models
models = build_models(random_state=42, num_cores=globals().get("num_cores", -1))
print("Models defined:", list(models.keys()))

# Evaluate on both datasets
results_df = evaluate_on_both_datasets(
    datasets=datasets,
    target_col=target_col,
    horizons=HORIZONS,
    models=models,
    test_size=0.2,
    random_state=42,
)

display(results_df.sort_values(["dataset","mae"]).round(4))

# Compare MAE across datasets
mae_pivot = mae_comparison_table(results_df)
display(mae_pivot.round(4))


## Checking which station is the closest to Bern - Useless
So we can do 

In [ ]:
import re
from pathlib import Path

import requests
import certifi

STATION_META_URL = (
    "https://data.geo.admin.ch/ch.meteoschweiz.messnetz-automatisch/"
    "ch.meteoschweiz.messnetz-automatisch_en.csv"
)

# Store inside the repo (reproducible path)
STATION_META_PATH = Path("weather_stations_data/stations_meteoswiss.csv")
STATION_META_PATH.parent.mkdir(parents=True, exist_ok=True)


def fetch_station_meta(
    path: Path = STATION_META_PATH,
    url: str = STATION_META_URL,
    force: bool = False
) -> Path:
    """
    Download MeteoSwiss station metadata CSV if missing (or if force=True).
    Uses requests + certifi to avoid common macOS SSL issues.
    """
    if path.exists() and not force:
        return path

    r = requests.get(url, timeout=60, verify=certifi.where())
    r.raise_for_status()
    path.write_bytes(r.content)
    return path


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def station_codes_in_df(df):
    codes = set()
    for c in df.columns:
        m = re.search(r"_([A-Z]{3})$", c)
        if m:
            codes.add(m.group(1))
    return sorted(codes)


def pick_most_bern_like_station(
    df,
    station_meta_path: Path = STATION_META_PATH,
    weight_elev: float = 0.2
):
    # Ensure file exists (download if needed)
    station_meta_path = fetch_station_meta(station_meta_path)

    meta = pd.read_csv(station_meta_path, sep=";", encoding="latin-1")

    # Adjust these mappings if the CSV schema changes
    meta = meta.rename(columns={
        "Abbr.": "abbr",
        "Station height m a. sea level": "elev_m",
        "Latitude": "lat",
        "Longitude": "lon"
    })

    needed = {"abbr", "elev_m", "lat", "lon"}
    missing_cols = needed - set(meta.columns)
    if missing_cols:
        raise ValueError(
            f"Station metadata CSV missing columns: {missing_cols}. Got: {list(meta.columns)}"
        )

    codes = set(station_codes_in_df(df))
    meta = meta[meta["abbr"].isin(codes | {"BER"})].copy()

    if (meta["abbr"] == "BER").sum() == 0:
        raise ValueError("BER station not found in metadata CSV (check file/version).")

    ber = meta.loc[meta["abbr"] == "BER"].iloc[0]
    cand = meta.loc[meta["abbr"] != "BER"].copy()

    cand["dist_km"] = haversine_km(ber["lat"], ber["lon"], cand["lat"], cand["lon"])
    cand["elev_diff_m"] = (cand["elev_m"] - ber["elev_m"]).abs()
    cand["score"] = cand["dist_km"] + weight_elev * cand["elev_diff_m"]

    cand_sorted = cand.sort_values("score")
    best = cand_sorted.iloc[0]
    shortlist = cand_sorted[["abbr", "dist_km", "elev_m", "elev_diff_m", "score"]].head(10)

    return best["abbr"], shortlist


# --- Usage ---
best_station, shortlist = pick_most_bern_like_station(df24_imputed)
print("Bern-like station in your columns:", best_station)
display(shortlist)


In [ ]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(max_iter=5000),
    "RandomForest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        criterion="absolute_error"
    ),
    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "KNN": KNeighborsRegressor(),
    "SVR_rbf": SVR(kernel="rbf"),
    "MLP": MLPRegressor(random_state=42, max_iter=300),
}

# Optional LightGBM / CatBoost if available
if "lgb_available" in globals() and lgb_available:
    models["LGBMRegressor"] = lgb.LGBMRegressor(
        n_estimators=500,
        objective="regression",
        random_state=42
    )

if "cb_available" in globals() and cb_available:
    models["CatBoostRegressor"] = cb.CatBoostRegressor(
        verbose=0,
        random_state=42,
        loss_function="MAE"
    )


In [ ]:
current_horizon = "24h"
target_col = horizons[current_horizon]

datasets_24h = {
    "drop_na": df24_dropna.copy(),
    "imputed": df24_imputed.copy()
}

baseline_results = evaluate_on_both_datasets(
    datasets=datasets_24h,
    target_col=target_col,
    horizons=horizons,
    models=models,
    test_size=0.2,
    random_state=42
)

datasets_24h_noINT = {
    name: drop_station_predictors(df, "INT", horizons)
    for name, df in datasets_24h.items()
}

noINT_results = evaluate_on_both_datasets(
    datasets=datasets_24h_noINT,
    target_col=target_col,
    horizons=horizons,
    models=models,
    test_size=0.2,
    random_state=42
)

baseline_results["station_removed"] = "NONE"
noINT_results["station_removed"] = "INT"

combined = pd.concat([baseline_results, noINT_results], ignore_index=True)

# MAE deltas (INT removed vs baseline) per model + dataset
pivot = combined.pivot_table(
    index=["model", "dataset"],
    columns="station_removed",
    values="mae"
).reset_index()

pivot["mae_delta_NO_INT"] = pivot["INT"] - pivot["NONE"]
display(pivot.sort_values("mae_delta_NO_INT", ascending=False).round(4))


# Discretizar precipitación
si una variable es todo cero, menos el 3%, el modelo va a creer que todo es cero, por eficiencia de procesamiento.

In [ ]:
# Raw precipitation columns
precipitation_columns = [
    col for col in train.columns
    if col.startswith('rre150h0_')
]

# Sanity check
assert precipitation_columns, "No precipitation columns found"

# Binning setup
bins = [0, 1, 5, 10, 20, np.inf]
labels = ['No rain', 'Very light', 'Light', 'Moderate', 'Heavy']

for col in precipitation_columns:
    # Ensure numeric
    train[col] = pd.to_numeric(train[col], errors='coerce')

    # Create bins
    train[f'{col}_bins'] = pd.cut(
        train[col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# All binned precipitation columns
bin_columns = [
    col for col in train.columns
    if col.startswith('rre150h0_') and col.endswith('_bins')
]

# Stack all bins into a single Series
all_precip_bins = (
    train[bin_columns]
    .stack()
)


In [ ]:
[c for c in train.columns if 'rre150h0' in c]


In [ ]:
[col for col in train.columns if col.endswith('_bins')][:10]


# TAIL-STABILISED FEATURES (log1p)

In [ ]:


# Precipitation (very skewed, many zeros)
if "precip_total" in df.columns:
    df["precip_total_log1p"] = np.log1p(df["precip_total"])

if "precip_max" in df.columns:
    df["precip_max_log1p"] = np.log1p(df["precip_max"])

# Radiation (skewed, daytime spikes)
if "radiation_mean" in df.columns:
    df["radiation_mean_log1p"] = np.log1p(df["radiation_mean"])

if "radiation_max" in df.columns:
    df["radiation_max_log1p"] = np.log1p(df["radiation_max"])

# Wind (moderately skewed)
if "wind_mean_current" in df.columns:
    df["wind_mean_current_log1p"] = np.log1p(df["wind_mean_current"])

if "wind_max_current" in df.columns:
    df["wind_max_current_log1p"] = np.log1p(df["wind_max_current"])


# 4. Feature engineering: advanced weather features v2

In [ ]:
def create_advanced_features(df):
    """
    Lean, robust feature engineering for weather prediction.
    Designed to minimise overfitting and work well with Ridge.
    """
    df = df.copy()

    # ============================================================
    # 1. CYCLICAL TIME FEATURES
    # ============================================================
    # Hour -> cyclical encoding
    if "hour" in df.columns:
        df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
        df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
        df.drop(columns=["hour"], inplace=True)

    # Season -> one-hot encoding
    if "season" in df.columns:
        # normalise strings (handles "Winter", " WINTER ", etc.)
        df["season"] = df["season"].astype(str).str.strip().str.lower()

        # one-hot (drop_first avoids perfect collinearity for linear models)
        df = pd.get_dummies(df, columns=["season"], prefix="season", drop_first=True)

    # ============================================================
    # 2. TEMPERATURE (core signal)
    # ============================================================
    temp_cols = [c for c in df.columns if c.startswith("tre200h0_") and "lag" not in c]

    if temp_cols:
        df["temp_mean"] = df[temp_cols].mean(axis=1)

    if "tre200h0" in df.columns and "temp_mean" in df.columns:
        df["temp_anomaly"] = df["tre200h0"] - df["temp_mean"]

    if "tre200h0" in df.columns and "tre200h0_lag24h" in df.columns:
        df["temp_change_24h"] = df["tre200h0"] - df["tre200h0_lag24h"]

    # ============================================================
    # 3. HUMIDITY
    # ============================================================
    hum_cols = [c for c in df.columns if c.startswith("ure200h0_")]

    if hum_cols:
        df["humidity_mean"] = df[hum_cols].mean(axis=1)

    # ============================================================
    # 4. PRESSURE
    # ============================================================
    pres_cols = [c for c in df.columns if c.startswith("prestah0_")]

    if pres_cols:
        df["pressure_mean"] = df[pres_cols].mean(axis=1)

    # ============================================================
    # 5. WIND
    # ============================================================
    wind_cols = [c for c in df.columns if c.startswith("fkl010h0_")]

    if wind_cols:
        df["wind_mean"] = df[wind_cols].mean(axis=1)

    # ============================================================
    # 6. PRECIPITATION (heavy-tailed)
    # ============================================================
    precip_cols = [c for c in df.columns if c.startswith("rre150h0_")]

    if precip_cols:
        df["precip_total"] = df[precip_cols].sum(axis=1)
        df["precip_total_log1p"] = np.log1p(df["precip_total"])

    # ============================================================
    # 7. RADIATION (heavy-tailed)
    # ============================================================
    rad_cols = [c for c in df.columns if c.startswith("gre000h0_")]

    if rad_cols:
        df["radiation_mean"] = df[rad_cols].mean(axis=1)
        df["radiation_mean_log1p"] = np.log1p(df["radiation_mean"])

    # ============================================================
    # 8. DROP RAW STATION COLUMNS
    # ============================================================
    drop_cols = temp_cols + hum_cols + pres_cols + wind_cols + precip_cols + rad_cols
    df.drop(columns=drop_cols, inplace=True, errors="ignore")

    return df


# 5. Building dataset variants for modelling

## 5.1 Outliser removal

HERE OUTLIER REMOVAL¡¡¡

## 5.2 Creation of the datasets

In [ ]:
# 1) Apply advanced feature engineering to the raw train data
train_fe = create_advanced_features(train, is_training=True)

# If you have an outlier-removal step, apply it HERE on train_fe
# e.g. train_clean = remove_outliers(train_fe)
# For now, just copy:
train_clean = train_fe.copy()

print("Original train shape:", train.shape)
print("After feature engineering (train_fe):", train_fe.shape)
print("After optional cleaning (train_clean):", train_clean.shape)

# --- Step 1: Preserve original and define base dataset ---
base = train_clean.copy()

# Identify numeric and categorical columns in the dataset
num_cols = base.select_dtypes(include="number").columns
cat_cols = base.select_dtypes(exclude="number").columns

print(f"Numeric columns: {len(num_cols)}, Categorical columns: {len(cat_cols)}")

# 1) Drop rows with any NA values  → main 'drop_na' dataset
train_drop = base.dropna().copy()

# 2) Mean-imputed dataset
mean_imputer = SimpleImputer(strategy="mean")
train_mean_imp = base.copy()
train_mean_imp[num_cols] = mean_imputer.fit_transform(train_mean_imp[num_cols])

if len(cat_cols) > 0:
    cat_modes_mean = train_mean_imp[cat_cols].mode().iloc[0]
    train_mean_imp[cat_cols] = train_mean_imp[cat_cols].fillna(cat_modes_mean)

# 3) Median-imputed dataset
median_imputer = SimpleImputer(strategy="median")
train_median_imp = base.copy()
train_median_imp[num_cols] = median_imputer.fit_transform(train_median_imp[num_cols])

if len(cat_cols) > 0:
    cat_modes_median = train_median_imp[cat_cols].mode().iloc[0]
    train_median_imp[cat_cols] = train_median_imp[cat_cols].fillna(cat_modes_median)

# 4) KNN imputation (more sophisticated)
print("\nPerforming KNN imputation (this may take a moment)...")
train_knn_imp = base.copy()

knn_numeric_cols = train_knn_imp.select_dtypes(include=[np.number]).columns
knn_numeric_data = train_knn_imp[knn_numeric_cols]

knn_imputer = KNNImputer(n_neighbors=5, weights="distance")
train_knn_imp[knn_numeric_cols] = knn_imputer.fit_transform(knn_numeric_data)

if len(cat_cols) > 0:
    cat_modes_knn = train_knn_imp[cat_cols].mode().iloc[0]
    train_knn_imp[cat_cols] = train_knn_imp[cat_cols].fillna(cat_modes_knn)

print(f"\nknn_imputed shape: {train_knn_imp.shape}")
print("\nAll dataset variants created successfully!")

# Sanity checks
print(f"\nRows before drop: {len(base)}, after drop: {len(train_drop)}")
print("Remaining NAs after drop:", train_drop.isnull().sum().sum())
print("Remaining NAs after mean imputation:", train_mean_imp.isnull().sum().sum())
print("Remaining NAs after median imputation:", train_median_imp.isnull().sum().sum())
print("Remaining NAs after KNN imputation:", train_knn_imp.isnull().sum().sum())

# --- Step 2: Build datasets dictionary (NO scaling, NO PCA) ---
datasets = {
    "drop_na":        train_drop,
    "mean_imputed":   train_mean_imp,
    "median_imputed": train_median_imp,
    "knn_imputed":    train_knn_imp,
}

# Quick sanity check on cyclical features in the main dataset
if {"hour_sin", "hour_cos"}.issubset(datasets["drop_na"].columns):
    display(datasets["drop_na"][["hour_sin", "hour_cos"]].head())
